# 04 — Energy and Decay

In notebook 03, the ball bounced forever — same height every time. Mathematically perfect. Physically wrong.

Real balls lose energy. A basketball bounces lower on each impact and settles to rest. A rubber ball skips a few times and stops.

One small change makes it feel real:

```python
dy = -dy * 0.8
```

That `0.8` is not arbitrary. This notebook explains what it is, why multiplying by the same number repeatedly produces a specific curve, and where that curve appears throughout physics and machine learning.

**Prerequisites:** [03 — Bouncing](./03-bouncing.ipynb). The coefficient of restitution and the quadratic arc formula.

---

## 1. One Change Makes It Real

In notebook 03, the bounce was `dy = -dy`. The ball returned to exactly the height it fell from.

Replace the bounce line with:

```python
dy = -dy * 0.8
```

The ball now exits each impact at 80% of the speed it entered. The arcs shrink. Watch the `dy` label — its magnitude after each bounce is 80% of what it was before.

In [ ]:
# FuncAnimation requires an interactive backend — add '%matplotlib widget' above if it renders static
import matplotlib.pyplot as plt
import matplotlib.animation as animation

x0, y0     = 5.0, 26.0
dx0        = 0.7
gravity    = -0.15
floor_y    = 4.0
restitution = 0.8     # was 1.0 in notebook 03
friction    = 0.99    # horizontal speed multiplier per frame
clamp_v     = 0.4     # stop bouncing when |dy| falls below this
FRAMES      = 200

# Pre-simulate the full trajectory
x, y, dy, dx = x0, y0, 0.0, dx0
all_positions  = []
bounce_events  = []   # (x_pos, frame, |dy_before|, |dy_after|)
peak_events    = []   # (frame, peak_y) — detected when dy crosses zero downward
stopped        = False

prev_dy = dy
for f in range(FRAMES):
    prev_dy = dy
    dy  += gravity
    y   += dy
    dx  *= friction
    x   += dx

    # Detect peak (dy just changed sign from positive to negative)
    if prev_dy > 0 and dy <= 0 and not stopped:
        peak_events.append((f, y))

    if y <= floor_y and not stopped:
        v_before = abs(dy)
        y   = floor_y
        dy  = -dy * restitution
        v_after = abs(dy)
        bounce_events.append((x, f, v_before, v_after))
        if abs(dy) < clamp_v:
            dy = 0.0
            stopped = True

    all_positions.append((x, y, dy, dx))

# --- Build the figure ---
fig, ax = plt.subplots(figsize=(10, 3))
fig.patch.set_facecolor('#1e1e2e')
ax.set_facecolor('#1e1e2e')
ax.set_xlim(0, 70)
ax.set_ylim(0, 30)
ax.set_aspect('equal')
ax.axis('off')

ax.axhline(floor_y, color='#45475a', linewidth=1.5)
ax.text(1, floor_y + 0.7, 'floor', color='#6c7086', fontsize=8, fontfamily='monospace')

# Static bounce markers with speed labels
for i, (bx, _, v_before, v_after) in enumerate(bounce_events, 1):
    ax.plot(bx, floor_y, 'v', color='#f38ba8', markersize=6, zorder=5)
    ax.text(bx, floor_y - 2.5, f'{v_after:.2f}', color='#f38ba8',
            fontsize=7, ha='center', fontfamily='monospace')

# Speed-after label header
ax.text(all_positions[-1][0] + 1.5, floor_y - 2.5, '← |dy| after',
        color='#6c7086', fontsize=7, fontfamily='monospace')

ball = plt.Circle((x0, y0), radius=3, color='#89dceb')
ax.add_patch(ball)
trail_dots = [plt.Circle((0, 0), radius=0.8, color='#89dceb', alpha=0) for _ in range(20)]
for dot in trail_dots:
    ax.add_patch(dot)

info = ax.text(2, 28.5, '', color='#cdd6f4', fontsize=9, fontfamily='monospace')

def update(frame):
    px, py, pdy, pdx = all_positions[frame]
    ball.set_center((px, py))
    for i, dot in enumerate(trail_dots):
        idx = frame - len(trail_dots) + i + 1
        if 0 <= idx < len(all_positions):
            dot.set_center(all_positions[idx][:2])
            dot.set_alpha(0.04 + 0.045 * i)
        else:
            dot.set_alpha(0)
    info.set_text(f't={frame:>3}   dy={pdy:+.2f}   dx={pdx:.2f}')
    return [ball, info] + trail_dots

ani = animation.FuncAnimation(fig, update, frames=FRAMES, interval=60, blit=True)
plt.tight_layout()
plt.show()

# Print what happened at each bounce
print(f'Bounce  |dy| before  |dy| after  ratio')
print('-' * 45)
for i, (_, _, v_before, v_after) in enumerate(bounce_events, 1):
    ratio = v_after / v_before if v_before > 0 else 0
    print(f'{i:>6}  {v_before:>10.3f}  {v_after:>9.3f}  {ratio:.3f}')

---

## 2. The Restitution Coefficient

The number `0.8` in `dy = -dy * 0.8` is the **coefficient of restitution**, written $e$.

$$e = \frac{|v_{\text{after}}|}{|v_{\text{before}}|}$$

It is the ratio of outgoing speed to incoming speed at the moment of impact. Three cases cover all physical possibilities:

| $e$ | Meaning | Example |
|---|---|---|
| $e = 1$ | Perfectly elastic — speed fully preserved | Superball, notebook 03 |
| $0 < e < 1$ | Inelastic — speed reduced | Tennis ball ($≈0.73$), basketball ($≈0.85$) |
| $e = 0$ | Perfectly inelastic — stops dead at impact | Ball of clay |

In code:

```python
dy = -dy * e   # e = 1 is notebook 03; e = 0 stops the ball; anything between is physical
```

Programmer framing: $e$ is a **decay factor** applied at each bounce event. After $n$ bounces, the vertical speed at impact is $e^n$ times the initial impact speed. The factor is applied discretely — once per bounce, not continuously.

In [ ]:
import matplotlib.pyplot as plt

def simulate_decay(y0, gravity, floor, e, frames, clamp=0.05):
    dy, y = 0.0, float(y0)
    ys = [y]
    for _ in range(frames):
        dy += gravity
        y  += dy
        if y <= floor:
            y  = floor
            dy = -dy * e
            if abs(dy) < clamp:
                dy = 0.0
        ys.append(y)
    return ys

configs = [
    (1.0, '#cdd6f4', 'e = 1.0  (notebook 03 — no decay)'),
    (0.9, '#a6e3a1', 'e = 0.9  (slow decay)'),
    (0.7, '#fab387', 'e = 0.7  (medium decay)'),
    (0.5, '#f38ba8', 'e = 0.5  (fast decay)'),
]

fig, ax = plt.subplots(figsize=(11, 4))
fig.patch.set_facecolor('#1e1e2e')
ax.set_facecolor('#1e1e2e')
ax.tick_params(colors='#cdd6f4')
for spine in ax.spines.values():
    spine.set_edgecolor('#45475a')

for e_val, color, label in configs:
    ys = simulate_decay(y0=26, gravity=-0.15, floor=4, e=e_val, frames=200)
    ax.plot(ys, color=color, linewidth=1.6, label=label, alpha=0.9)

ax.axhline(4, color='#45475a', linewidth=1.2)
ax.set_xlabel('t  (frame)', color='#cdd6f4')
ax.set_ylabel('y  (position)', color='#cdd6f4')
ax.set_title('Four values of e — same equation, different boundary condition', color='#cdd6f4', pad=12)
ax.legend(facecolor='#313244', labelcolor='#cdd6f4', framealpha=0.85, fontsize=9)
plt.tight_layout()
plt.show()

---

## 3. Exponential Decay

Apply the same multiplier $e$ at each bounce. The impact speed after $n$ bounces is:

$$v_n = e^n \cdot v_0$$

**Where the height formula comes from:**

At the peak of each arc, all kinetic energy has converted to potential energy: $\frac{1}{2}mv^2 = mgh$, so $h = v^2/(2|g|)$. Height is proportional to $v^2$.

If velocity decays by $e$ per bounce, height decays by $e^2$:

$$h_n = \frac{v_n^2}{2|g|} = \frac{(e^n \cdot v_0)^2}{2|g|} = e^{2n} \cdot \underbrace{\frac{v_0^2}{2|g|}}_{h_0}$$

$$\boxed{h_n = h_0 \cdot e^{2n}}$$

For $e = 0.8$: each bounce reaches $0.8^2 = 0.64$ of the previous height — 64% every bounce.

In code, this is just multiplying by the same number in a loop:

```python
h = h0
for n in range(num_bounces):
    h *= e**2   # this loop IS the formula h_n = h_0 * e^(2n)
```

---

**The same pattern appears everywhere:**

| Application | Formula | Decays by per step |
|---|---|---|
| Bounce height | $h_n = h_0 \cdot e^{2n}$ | $e^2$ per bounce |
| Horizontal friction | $dx_t = dx_0 \cdot f^t$ | $f$ per frame |
| Radioactive decay | $N(t) = N_0 \cdot (\tfrac{1}{2})^{t/T}$ | $\tfrac{1}{2}$ per half-life |
| ML weight decay | $w_n = w_0 \cdot (1-\lambda)^n$ | $(1-\lambda)$ per gradient step |
| Capacitor discharge | $V(t) = V_0 \cdot \exp(-t/RC)$ | depends on $RC$ |

All have the same form: $f(n) = A \cdot r^n$ for some starting value $A$ and ratio $0 < r < 1$.

> **Notation warning:** in $\exp(-t/RC)$, the $e$ is Euler's number $\approx 2.718$ — a mathematical constant. In $e = v_{\text{after}}/v_{\text{before}}$, $e$ is the restitution coefficient — a physical property. Same letter, different quantities. Context resolves the ambiguity, but watch for it.

In [ ]:
import math

# Show the decay table for e=0.8, h0=22
e_r = 0.8
h0  = 22.0

print(f'Coefficient of restitution e = {e_r}   (e² = {e_r**2})')
print(f'Starting height h₀ = {h0}')
print()
print(f'{"Bounce n":>10}  {"h_n = h0 × e^(2n)":>22}  {"fraction of h0":>16}  {"loop"}')
print('-' * 68)
h = h0
for n in range(9):
    formula = h0 * (e_r ** (2 * n))
    print(f'{n:>10}  {formula:>22.4f}  {formula/h0:>16.4f}  {h:.4f}')
    h *= e_r**2   # the loop version: same result

print()
print('Loop and formula agree: multiplying by e² each step IS the formula.')
print()

# Cross-domain connections: all the same pattern
print('The same A × rⁿ pattern in other fields:')
print()
n_steps = 10

def decay_table(label, A, r, steps):
    vals = [A * r**n for n in range(steps)]
    print(f'  {label}')
    print(f'  n = 0..{steps-1}: ' + '  '.join(f'{v:.2f}' for v in vals))
    print()

decay_table('Bounce height   h_n = 22 × 0.64^n         (r = 0.64 = 0.8²)', 22.0,  0.64, n_steps)
decay_table('Radioactive     N(t) = 1000 × 0.5^(t/T)   (half per period)',  1000.0, 0.5,  n_steps)
decay_table('Weight decay    w_n = 1.0 × (1-0.1)^n     (λ=0.1 per step)',   1.0,   0.9,  n_steps)
decay_table('Capacitor       V(t) = 5.0 × exp(-0.2)^t  (RC=5)',             5.0,   math.exp(-0.2), n_steps)

---

## 4. Horizontal Friction

The restitution coefficient acts at discrete events — once per bounce. Friction acts **every frame**:

```python
dx *= 0.99   # applied every frame, not just at bounces
```

After $t$ frames:

$$dx(t) = dx_0 \times 0.99^t$$

This is the same exponential pattern as restitution, but applied continuously. The multiplier feels negligibly small — 1% reduction per frame. The accumulation does not feel small:

$$0.99^{100} \approx 0.366 \qquad 0.99^{200} \approx 0.134 \qquad 0.99^{460} \approx 0.010$$

After 100 frames: horizontal speed is a third of its starting value. After 460 frames: it is 1% of its starting value.

**Continuous vs. discrete decay:**

| | Restitution | Friction |
|---|---|---|
| When applied | once per bounce | once per frame |
| Code | `dy = -dy * e` | `dx *= f` |
| Effect | discrete step down in speed | smooth exponential curve |
| Analogy | compound interest applied yearly | applied continuously |

The math is identical — multiply by a ratio less than 1, repeatedly. The difference is only in the frequency of application.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dx0     = 3.0
t_vals  = np.arange(0, 500)

friction_configs = [
    (0.99,  '#89dceb', 'f = 0.99  (1% per frame — typical game friction)'),
    (0.98,  '#a6e3a1', 'f = 0.98  (2% per frame — rougher surface)'),
    (0.95,  '#fab387', 'f = 0.95  (5% per frame — heavy drag)'),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#1e1e2e')

for ax in axes:
    ax.set_facecolor('#1e1e2e')
    ax.tick_params(colors='#cdd6f4')
    for spine in ax.spines.values():
        spine.set_edgecolor('#45475a')

for f_val, color, label in friction_configs:
    dx_t = dx0 * f_val ** t_vals
    axes[0].plot(t_vals, dx_t, color=color, linewidth=1.8, label=label)
    axes[1].semilogy(t_vals, dx_t / dx0, color=color, linewidth=1.8)

# Mark key thresholds on the linear plot
for thresh, label in [(0.5, '50%'), (0.1, '10%'), (0.01, '1%')]:
    axes[0].axhline(dx0 * thresh, color='#45475a', linewidth=0.7, linestyle=':')
    axes[0].text(5, dx0 * thresh + 0.04, label, color='#6c7086', fontsize=8)

axes[0].set_xlabel('t  (frames)', color='#cdd6f4')
axes[0].set_ylabel('dx  (horizontal speed)', color='#cdd6f4')
axes[0].set_title('Horizontal speed decay (linear scale)', color='#cdd6f4', pad=10)
axes[0].legend(facecolor='#313244', labelcolor='#cdd6f4', framealpha=0.85, fontsize=8)

axes[1].set_xlabel('t  (frames)', color='#cdd6f4')
axes[1].set_ylabel('dx / dx₀  (fraction of starting speed)', color='#cdd6f4')
axes[1].set_title('Log scale — straight lines confirm exponential decay', color='#cdd6f4', pad=10)
axes[1].tick_params(axis='y', colors='#cdd6f4')

plt.tight_layout()
plt.show()

# Print where the 50% and 1% crossings occur for f=0.99
f = 0.99
t_half = math.log(0.5)  / math.log(f)
t_cent = math.log(0.01) / math.log(f)
print(f'For friction = {f}:')
print(f'  50% of starting speed at frame {t_half:.0f}  (formula: log(0.5)/log({f}))')
print(f'   1% of starting speed at frame {t_cent:.0f}  (formula: log(0.01)/log({f}))')
print()
print('General rule: fraction r reached at frame t = log(r) / log(friction)')

---

## 5. Zeno's Paradox — and Why the Ball Actually Stops

With $e < 1$, the ball bounces infinitely many times — each arc smaller, but always a next one. This is the ancient puzzle: if motion is divided into infinitely many steps, can it ever complete?

Yes — because the infinite sum of bounce durations converges to a finite number.

Let $v_0$ be the impact speed at the first bounce. Each subsequent arc takes time $2v_n/|g| = 2e^n v_0 / |g|$ (up and back down). The total duration of all bounces after the first impact:

$$T_{\text{bounces}} = \frac{2v_0}{|g|} \sum_{n=0}^{\infty} e^n = \frac{2v_0}{|g|} \cdot \frac{1}{1-e}$$

Adding the initial drop time $t_0 = v_0/|g|$, the ball reaches permanent rest at:

$$T_{\text{total}} = \frac{v_0}{|g|} + \frac{2v_0}{|g|(1-e)} = \frac{v_0}{|g|} \cdot \frac{1+e}{1-e}$$

For $e = 0.8$: $T = v_0/|g| \times 9$. **Finite.** The infinite sequence of bounces completes in finite time — the paradox resolves because the arcs shrink fast enough for the sum to converge.

**In code:** we still clamp below a threshold. Not because the math breaks down, but because below a certain height the ball is physically indistinguishable from rest, and the simulation would spend thousands of frames on sub-pixel arcs:

```python
if abs(dy) < threshold:
    dy = 0.0
```

This is a physically motivated approximation, not a numerical hack.

In [ ]:
import math

# Demonstrate convergence: sum of bounce durations is finite
# Using the continuous formula for clarity

abs_g = 2.0   # |gravity|
h0    = 36.0  # starting height above floor
e_r   = 0.8

v0 = math.sqrt(2 * abs_g * h0)   # impact speed at first bounce

# Exact formula for total time
T_exact = (v0 / abs_g) * (1 + e_r) / (1 - e_r)

# Running sum: accumulate bounce durations
print(f'v₀ = {v0:.4f},  |g| = {abs_g},  e = {e_r}')
print(f'Exact total time = v₀/|g| × (1+e)/(1−e) = {v0/abs_g:.4f} × {(1+e_r)/(1-e_r):.1f} = {T_exact:.4f}')
print()
print(f'{"Bounce n":>10}  {"arc duration":>14}  {"cumulative time":>16}  {"remaining":>12}')
print('-' * 58)

t_drop = v0 / abs_g    # initial drop
t_cum  = t_drop
print(f'{"(drop)":>10}  {t_drop:>14.4f}  {t_cum:>16.4f}  {T_exact - t_cum:>12.4f}')

v = v0
for n in range(15):
    v *= e_r
    arc_time = 2 * v / abs_g
    t_cum   += arc_time
    remaining = T_exact - t_cum
    print(f'{n:>10}  {arc_time:>14.6f}  {t_cum:>16.4f}  {remaining:>12.6f}')

print(f'{"(...)":>10}  {"...":>14}  {T_exact:>16.4f}  {0:>12.6f}')
print()
print(f'After 15 bounces the cumulative time is within {T_exact - t_cum:.4f} of the exact total.')
print('The remaining difference = sum of all remaining (infinitely many) arcs.')
print()

# Show the clamping threshold in context
print('Speed after n bounces (v₀ × eⁿ):')
for n in range(20):
    vn = v0 * e_r**n
    arc_h = vn**2 / (2 * abs_g)
    if arc_h < 0.001:
        print(f'  n={n}: v={vn:.5f}, arc height={arc_h:.6f}  ← sub-millimetre, clamp here')
        break
    if n % 3 == 0:
        print(f'  n={n}: v={vn:.4f}, arc height={arc_h:.4f}')

---

## 6. The Decay Envelope

Run the simulation and record the peak height of each bounce. Those peaks follow $h_n = h_0 \times e^{2n}$ — the **exponential decay envelope**.

Plotting on a log scale turns the exponential into a straight line. This is the standard way to identify and verify exponential decay: if the log-scale plot is a straight line, the decay is exponential.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

def simulate_with_peaks(y0, gravity, floor, e_r, clamp=0.05, max_frames=2000):
    """Run simulation and return (all y positions, peak heights above floor per bounce arc)."""
    dy, y = 0.0, float(y0)
    ys, peak_heights = [y], []
    bounce_frames = []

    for f in range(max_frames):
        dy += gravity
        y  += dy
        if y <= floor:
            y  = floor
            dy = -dy * e_r
            bounce_frames.append(f + 1)
            if abs(dy) < clamp:
                dy = 0.0
        ys.append(y)
        if dy == 0.0:
            break

    # Extract peak heights: max y between consecutive bounce frames
    prev_bf = 0
    for bf in bounce_frames:
        segment = ys[prev_bf:bf]
        if segment:
            peak_heights.append(max(segment) - floor)
        prev_bf = bf

    return np.array(ys), np.array(peak_heights), bounce_frames

e_r    = 0.8
y0     = 26.0
floor  = 4.0
h0_sim = y0 - floor   # = 22

ys, peak_heights, bounce_frames = simulate_with_peaks(y0, gravity=-0.15, floor=floor, e_r=e_r)

n_bounces  = len(peak_heights)
ns         = np.arange(n_bounces)
h_formula  = h0_sim * (e_r ** (2 * ns))   # theoretical envelope

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#1e1e2e')

def style(ax, title):
    ax.set_facecolor('#1e1e2e')
    ax.tick_params(colors='#cdd6f4')
    for spine in ax.spines.values():
        spine.set_edgecolor('#45475a')
    ax.set_title(title, color='#cdd6f4', pad=10)
    ax.set_xlabel('Bounce number n', color='#cdd6f4')

# Left: linear scale
style(axes[0], 'Peak heights — linear scale')
axes[0].plot(ns, h_formula,  color='#f38ba8', linewidth=2,   linestyle='--',
             label=f'h₀ × e²ⁿ  (e={e_r}, h₀={h0_sim})')
axes[0].plot(ns, peak_heights, 'o', color='#89dceb', markersize=7, zorder=5,
             label='Simulation peaks')
axes[0].set_ylabel('Peak height above floor', color='#cdd6f4')
axes[0].legend(facecolor='#313244', labelcolor='#cdd6f4', framealpha=0.85, fontsize=9)

# Right: log scale — exponential becomes a straight line
style(axes[1], 'Peak heights — log scale (exponential = straight line)')
# Avoid log(0): only plot where peak_heights > 0
mask = peak_heights > 1e-6
axes[1].semilogy(ns[mask], h_formula[mask],    color='#f38ba8', linewidth=2,
                 linestyle='--', label=f'h₀ × e²ⁿ  (theory)')
axes[1].semilogy(ns[mask], peak_heights[mask], 'o', color='#89dceb', markersize=7, zorder=5,
                 label='Simulation')
axes[1].set_ylabel('Peak height (log scale)', color='#cdd6f4')
axes[1].tick_params(axis='y', colors='#cdd6f4')
axes[1].legend(facecolor='#313244', labelcolor='#cdd6f4', framealpha=0.85, fontsize=9)

plt.tight_layout()
plt.show()

# Print the table
print(f'Peak heights (e={e_r}, h₀={h0_sim}):')
print(f'{"n":>5}  {"simulation":>12}  {"formula h₀×e²ⁿ":>18}  {"ratio (e²)"}' )
print('-' * 52)
for i, (sim_h, form_h) in enumerate(zip(peak_heights, h_formula)):
    ratio = sim_h / peak_heights[i-1] if i > 0 and peak_heights[i-1] > 0 else float('nan')
    print(f'{i:>5}  {sim_h:>12.4f}  {form_h:>18.4f}  {ratio:>10.4f}')

print(f'\nExpected ratio per bounce: e² = {e_r**2:.4f}')
print('(Small deviations are Euler error at the bounce point, as discussed in notebook 02.)')

---

## Exercises

- **Try different values of $e$:** `0.9`, `0.5`, `1.0`, `0.0`. What does the trajectory look like for each? What does the decay envelope look like? At what value does the animation feel like a tennis ball? A basketball? A lump of clay?
- **Measure a real ball:** drop a ball and count how many times it bounces before you can barely see the arcs. Use $h_n = h_0 \times e^{2n}$ to estimate the restitution coefficient.
- **Vary friction:** try `dx *= 0.995`, `dx *= 0.98`, `dx *= 0.9`. How many frames until the ball is essentially stationary horizontally? Use `math.log(0.01) / math.log(friction)` to predict it before running the simulation.
- **Time to stop:** compute $T = v_0/|g| \times (1+e)/(1-e)$ for your simulation parameters. Run the simulation with a very small clamp threshold. Does the simulation stop at approximately $T$?
- **Gravity and restitution together:** what happens to the decay envelope if you double gravity? The arcs are shorter, but does $h_n = h_0 \times e^{2n}$ still hold? The formula for $h_n$ depends on $h_0$ but not directly on $g$ — why?

---

## Summary

| Concept | Plain English | Code | Math |
|---------|--------------|------|------|
| Restitution coefficient | fraction of speed kept on each bounce | `dy = -dy * e` | $e = |v_{\text{after}}| / |v_{\text{before}}|$ |
| Exponential decay | multiply by the same ratio repeatedly | loop: `h *= e**2` | $h_n = h_0 \cdot e^{2n}$ |
| Decay envelope | the curve containing all bounce peaks | peak heights per bounce | $h_n = h_0 \cdot e^{2n}$ |
| Continuous friction | decay applied every frame, not just at bounces | `dx *= f` each frame | $dx(t) = dx_0 \cdot f^t$ |
| Zeno's paradox | infinite bounces in finite time | clamp with `if abs(dy) < ε: dy = 0` | $T = v_0/|g| \cdot (1+e)/(1-e)$ |
| Log-linear relationship | exponential decay is a straight line on log scale | `plt.semilogy(...)` | $\log h_n = \log h_0 + 2n \log e$ |

**The through-line:** multiplying by the same number less than 1, repeatedly, always produces the same shape — the exponential decay curve. The coefficient of restitution, capacitor discharge, radioactive decay, and ML weight decay are all the same mathematical pattern at different scales. The programmer who writes `dy = -dy * 0.8` is, without knowing it, implementing one of the most ubiquitous patterns in quantitative science.

---

**Next:** [05 — Putting It Together](./05-putting-it-together.ipynb) — the complete bouncing ball, every line named and every formula gathered in one place.